# Estados de carga unitarios por nodo de piping (1 kgf en X, Y, Z)

Los **grupos** del modelo SAP tienen el nombre de los **nodos de piping** (análisis de flexibilidad
en CAESAR II) y están asignados al joint de SAP donde va la carga de piping. El nombre del grupo no
tiene por qué coincidir con el del joint.

Para cada nodo de piping con grupo en el `.s2k`:
1. Se crea un **estado de carga** (load pattern) con **el mismo nombre del grupo**.
2. Ese estado lleva una **carga de joint** de 1 kgf en X, Y y Z sobre el joint del grupo.

El notebook **no crea ni renombra grupos**: usa los que ya existen en el `.s2k`.

**Entradas** (datasets de Kaggle):
- `.s2k` base exportado desde SAP2000, en unidades kgf-m, con los grupos de nodos de piping asignados.
- Excel/CSV con la lista de nodos de piping: columna A = nombre del nodo (opcional, para verificar).

Los nodos sin grupo se saltan y se listan. Un grupo con más de un joint detiene la ejecución.

**Salida:** `/kaggle/working/<modelo>_NODOS.s2k` y `/kaggle/working/mapeo_nodos.csv`
(nodo de piping → joint de SAP). Cada ejecución parte del `.s2k` base, así que puedes repetirla.

In [ ]:
import re
import glob
from pathlib import Path

import pandas as pd

INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

MODEL_FILE = None   # ruta fija al .s2k; None = detectar automáticamente
NODES_FILE = None   # ruta fija al Excel/CSV de nodos; None = detectar automáticamente; "" = no usar

FORCE_KGF = (1.0, 1.0, 1.0)      # fuerza en kgf aplicada en X, Y, Z a cada estado
LOAD_CS = "GLOBAL"               # sistema de coordenadas de las cargas
ENCODING = "latin-1"             # preserva los bytes del .s2k al reescribirlo

In [ ]:
def find_single(exts):
    hits = sorted(p for p in glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
                  if Path(p).suffix.lower() in exts)
    if len(hits) != 1:
        raise FileNotFoundError(
            f"Se esperaba un archivo {sorted(exts)} en {INPUT_DIR}, encontrados: {hits}. "
            "Fija MODEL_FILE / NODES_FILE en la celda de configuración."
        )
    return hits[0]


model_path = MODEL_FILE or find_single({".s2k"})
if Path(model_path).stem.endswith("_NODOS"):
    raise ValueError("El .s2k de entrada es una salida de este notebook. Usa siempre el modelo base: cada ejecución parte de él.")
nodes_path = None if NODES_FILE == "" else (NODES_FILE or find_single({".xlsx", ".xls", ".csv"}))
print("Modelo:", model_path)
print("Nodos:", nodes_path or "(no se usa lista; se toman todos los grupos)")

In [ ]:
def normalize_name(v):
    s = str(v).strip()
    return s[:-2] if re.fullmatch(r"\d+\.0", s) else s


def read_nodes(path):
    """Lista de nodos de piping: columna A. Una fila de cabecera 'nodo' se ignora."""
    if path.lower().endswith(".csv"):
        raw = pd.read_csv(path, header=None, dtype=str)
    else:
        raw = pd.read_excel(path, header=None, dtype=str)
    nodes = raw.iloc[:, 0].dropna().map(normalize_name)
    if nodes.iloc[0].lower() in {"nodo", "nodos", "node"}:
        nodes = nodes.iloc[1:]
    dup = nodes[nodes.duplicated(keep=False)]
    if len(dup):
        raise ValueError(f"Nodos duplicados en la lista:\n{sorted(set(dup))}")
    return list(nodes)


nodes_list = read_nodes(nodes_path) if nodes_path else None
print(f"Nodos en la lista: {len(nodes_list)}" if nodes_list else "Sin lista de nodos.")

In [ ]:
TABLE_RE = re.compile(r'^TABLE:\s+"(.+?)"')
KV_RE = re.compile(r'(\w+)=("[^"]*"|\S+)')


def parse_s2k(path):
    with open(path, encoding=ENCODING) as f:
        lines = f.read().splitlines()
    tables = {}
    current = None
    for line in lines:
        m = TABLE_RE.match(line.strip())
        if m:
            current = m.group(1)
            tables.setdefault(current, [])
        elif current is not None and "=" in line:
            tables[current].append({k: v.strip('"') for k, v in KV_RE.findall(line)})
    return lines, tables


def group_members(tables):
    """{grupo: {joints}} a partir de la tabla de asignaciones de grupos (solo joints)."""
    members = {}
    for name, rows in tables.items():
        if "GROUP" not in name.upper() or "ASSIGN" not in name.upper():
            continue
        for row in rows:
            if row.get("ObjectType", "").lower() == "joint":
                members.setdefault(row["GroupName"], set()).add(row["ObjectLabel"])
    return members


lines, tables = parse_s2k(model_path)

units = next((r["CurrUnits"] for r in tables.get("PROGRAM CONTROL", []) if "CurrUnits" in r), None)
if units is None:
    print("Aviso: no encontré CurrUnits en PROGRAM CONTROL; verifica a mano que el modelo esté en kgf-m.")
else:
    parts = [p.strip().lower() for p in units.split(",")]
    if parts[:2] != ["kgf", "m"]:
        raise ValueError(f"El modelo está en '{units}', no en kgf-m. Cambia unidades en SAP.")
    print("Unidades del modelo:", units)

members = group_members(tables)
defined_patterns = {r["LoadPat"] for r in tables.get("LOAD PATTERN DEFINITIONS", []) if "LoadPat" in r}
print(f"Grupos con joints: {len(members)}. Estados definidos: {len(defined_patterns)}")

In [ ]:
# Mapeo: el nombre del grupo es el nodo de piping; su único joint es el punto de carga en SAP.
groups = list(members) if nodes_list is None else nodes_list
mapping, sin_grupo, multi = {}, [], []
for node in groups:
    if node not in members:
        sin_grupo.append(node)
        continue
    js = sorted(members[node])
    if len(js) != 1:
        multi.append(f"grupo '{node}' tiene {len(js)} joints {js}")
        continue
    mapping[node] = js[0]

if multi:
    raise ValueError("Grupos con más de un joint (no se sabe a cuál aplicar la carga):\n  " + "\n  ".join(multi))

mapeo = pd.DataFrame([{"nodo_piping": n, "joint_sap": j} for n, j in mapping.items()],
                     columns=["nodo_piping", "joint_sap"])
print(f"Nodos con grupo y joint: {len(mapping)}. Sin grupo (se saltan): {len(sin_grupo)}")
if sin_grupo:
    print("  " + ", ".join(sin_grupo))
if not mapping:
    raise ValueError("Ningún nodo tiene grupo en el .s2k. Revisa los nombres de grupo.")
mapeo

In [ ]:
new_patterns = [n for n in mapping if n not in defined_patterns]
print(f"Estados nuevos: {len(new_patterns)}. Cargas de joint: {len(mapping)} (cada una {FORCE_KGF} kgf en X, Y, Z)")


def fmt(v):
    return f'"{v}"'


def num(v):
    return f"{float(v):.10g}"


def add_rows(lines, table_name, new_rows):
    """Inserta filas justo debajo de la cabecera de la tabla; si no existe, la crea antes de END TABLE DATA."""
    if not new_rows:
        return lines
    for i, line in enumerate(lines):
        m = TABLE_RE.match(line.strip())
        if m and m.group(1) == table_name:
            return lines[:i + 1] + new_rows + lines[i + 1:]
    block = ["", f'TABLE:  "{table_name}"'] + new_rows
    end = next((i for i, l in enumerate(lines) if l.strip() == "END TABLE DATA"), len(lines))
    return lines[:end] + block + lines[end:]


def replace_existing(lines, pairs):
    """Quita las cargas de joint ya existentes para los pares (joint, estado) de este run."""
    keep, replaced, nonzero, table = [], 0, 0, None
    for line in lines:
        m = TABLE_RE.match(line.strip())
        if m:
            table = m.group(1)
        elif table == "JOINT LOADS - FORCE" and "=" in line:
            row = {k: v.strip('"') for k, v in KV_RE.findall(line)}
            if (row.get("Joint"), row.get("LoadPat")) in pairs:
                replaced += 1
                if any(float(row.get(k, 0)) != 0 for k in ["F1", "F2", "F3", "M1", "M2", "M3"]):
                    nonzero += 1
                continue
        keep.append(line)
    print(f"Cargas existentes reemplazadas: {replaced} (de ellas {nonzero} distintas de cero)")
    return keep


fx, fy, fz = (num(v) for v in FORCE_KGF)
pattern_rows = [f"   LoadPat={fmt(p)}  DesignType=Other  SelfWtMult=0" for p in new_patterns]
joint_rows = [
    f"   Joint={fmt(mapping[n])}  LoadPat={fmt(n)}  CoordSys={LOAD_CS}  "
    f"F1={fx}  F2={fy}  F3={fz}  M1=0  M2=0  M3=0"
    for n in mapping
]

new_lines = replace_existing(lines, {(mapping[n], n) for n in mapping})
new_lines = add_rows(new_lines, "LOAD PATTERN DEFINITIONS", pattern_rows)
new_lines = add_rows(new_lines, "JOINT LOADS - FORCE", joint_rows)

out_path = Path(OUTPUT_DIR) / f"{Path(model_path).stem}_NODOS.s2k"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding=ENCODING, newline="\n") as f:
    f.write("\n".join(new_lines) + "\n")
mapeo.to_csv(Path(OUTPUT_DIR) / "mapeo_nodos.csv", index=False)
print(f"Escrito: {out_path}")
print(f"{len(joint_rows)} cargas de joint, {len(pattern_rows)} estados nuevos")

In [ ]:
_, out_tables = parse_s2k(out_path)
out_members = group_members(out_tables)
out_patterns = {r["LoadPat"] for r in out_tables["LOAD PATTERN DEFINITIONS"]}
out_loads = {(r["Joint"], r["LoadPat"]): r for r in out_tables["JOINT LOADS - FORCE"]}

for node, joint in mapping.items():
    assert out_members.get(node) == {joint}, f"el grupo {node} cambió de joints"
    assert node in out_patterns, f"falta el estado {node}"
    row = out_loads.get((joint, node))
    assert row is not None, f"falta la carga en joint {joint}, estado {node}"
    got = [float(row[k]) for k in ("F1", "F2", "F3")]
    assert all(abs(g - e) < 1e-9 for g, e in zip(got, FORCE_KGF)), f"carga incorrecta en {node}: {got}"
print(f"Verificación OK: {len(mapping)} nodos, cada uno con su estado y carga de {FORCE_KGF} kgf")